# 📚 Session 7 Homework: Build Your Own Knowledge Agent

**ISOM 260: AI for Business** | Suffolk University | Prof. Hasan Arslan

---

In class, your agent read **CloudSync Solutions'** documents. It could answer questions about pricing, refund policies, and support tiers — because we gave it those documents.

Now it reads **yours.**

You'll pick a topic you actually care about, load real information, and build a knowledge agent that becomes an expert in *your* domain. Same RAG engine from class — your content, your questions, your expertise.

### What You'll Do

| Task | What | Time |
|------|------|------|
| **Task 1: Build** | Pick a domain, load 5+ documents, test your agent | ~45 min |
| **Task 2: Break** | Find 3 failure modes (outdated data, contradictions, gaps) | ~30 min |
| **Task 3: Evaluate** | Score your agent on 10 test questions | ~30 min |
| **Task 4: Design** | Write a real-world knowledge agent proposal | ~20 min |

**Total: ~2 hours**

---

**Submission:** Run all cells so output is visible → Share your Colab link (Anyone with link can view) → Paste link in Canvas.

### A Note on Using AI Tools

You’re welcome to use ChatGPT, Claude, Copilot, or any AI tool to help you with this homework — for brainstorming domain ideas, drafting document content, writing test questions, or polishing your proposal.

That’s not a loophole. That’s the skill. Knowing how to use AI tools effectively is literally what this course teaches. The learning happens when you choose a domain, decide what documents matter, evaluate whether the agent’s answers are actually correct, and figure out where it breaks.

No AI tool can do that evaluation for you — that requires *your* judgment.

---

## 🚀 Setup (Just Run These)

Same setup as class. 2 cells, 2 minutes. You’ve done this before.

In [ ]:
# Install the Anthropic SDK
!pip install -q anthropic

In [ ]:
# Set your API key + verify connection
import os
import json
import anthropic
from google.colab import userdata

# Load API key from Colab Secrets (recommended)
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # <-- Replace if needed
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

client = anthropic.Anthropic()

# Quick connection test
response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'Homework notebook connected!' and nothing else."}]
)
print(f"🟢 {response.content[0].text}")
print(f"   Model: {response.model}")

---

## 🔧 Your RAG Toolkit (Pre-Built)

These are the same functions from class. **Just run all 4 cells** — don’t modify them.

| Cell | What It Does |
|------|--------------|
| `chunk_documents()` | Splits your documents into searchable passages |
| `search_documents()` | Finds the most relevant passages for a query (TF-IDF) |
| `search_my_docs()` | Wrapper + tool definition so Claude can call it |
| `run_rag_agent()` | The full ReAct agent loop from class |

In [ ]:
# ============================================
# DOCUMENT CHUNKING (from class)
# ============================================
# Splits documents into overlapping ~500-character chunks.
# Each chunk keeps its metadata so we know where it came from.

def chunk_documents(documents, chunk_size=500, overlap=100):
    """
    Split documents into overlapping chunks.

    Args:
        documents: List of dicts with 'title', 'content', 'source' fields
        chunk_size: Target size of each chunk in characters
        overlap: Number of overlapping characters between chunks

    Returns:
        List of chunk dicts with 'text', 'title', 'source', 'chunk_index' fields
    """
    chunks = []

    for doc in documents:
        content = doc['content']
        title = doc['title']
        source = doc['source']
        last_updated = doc.get('last_updated', 'Unknown')

        # Split into chunks with overlap
        start = 0
        chunk_index = 0

        while start < len(content):
            end = start + chunk_size

            # Try to break at a paragraph or sentence boundary
            if end < len(content):
                newline_pos = content.rfind('\n', start + chunk_size - 100, end + 50)
                if newline_pos > start:
                    end = newline_pos

            chunk_text = content[start:end].strip()

            if chunk_text:
                chunks.append({
                    'text': chunk_text,
                    'title': title,
                    'source': source,
                    'last_updated': last_updated,
                    'chunk_index': chunk_index
                })
                chunk_index += 1

            start = end - overlap if end < len(content) else len(content)

    return chunks

print("✅ chunk_documents() ready")

In [ ]:
# ============================================
# TF-IDF DOCUMENT SEARCH (from class)
# ============================================
# Scores each chunk by how relevant it is to a query.
# TF = how often a word appears in a chunk
# IDF = how rare that word is across ALL chunks

from collections import Counter
import math

def search_documents(query, chunks, top_k=3):
    """
    Search documents using TF-IDF scoring.
    Returns top_k most relevant (score, chunk) pairs.
    """
    stop_words = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'what', 'how',
        'do', 'does', 'i', 'my', 'our', 'their', 'for', 'to', 'and',
        'or', 'of', 'in', 'on', 'at', 'can', 'about', 'it', 'be',
        'that', 'this', 'with', 'from', 'not', 'but', 'if', 'when',
        'there', 'which', 'each', 'have', 'has', 'had', 'will', 'would'
    }

    query_terms = [w for w in query.lower().split() if w not in stop_words]
    if not query_terms:
        return []

    num_chunks = len(chunks)
    doc_freq = Counter()
    chunk_word_counts = []

    for chunk in chunks:
        words = chunk['text'].lower().split()
        word_count = Counter(words)
        chunk_word_counts.append(word_count)
        for term in set(words):
            doc_freq[term] += 1

    scored = []
    for idx, chunk in enumerate(chunks):
        word_count = chunk_word_counts[idx]
        total_words = sum(word_count.values()) or 1

        score = 0.0
        for term in query_terms:
            tf  = word_count[term] / total_words
            idf = math.log((num_chunks + 1) / (doc_freq.get(term, 0) + 1))
            score += tf * idf

        if score > 0:
            scored.append((round(score, 4), chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

print("✅ search_documents() ready — TF-IDF scoring")

In [ ]:
# ============================================
# SEARCH WRAPPER + TOOL DEFINITION
# ============================================
# This creates a search function and tool definition that reference
# the global `my_chunks` variable. You'll set that in Task 1.

def search_my_docs(query: str) -> str:
    """Search YOUR documents and return formatted results."""
    results = search_documents(query, my_chunks, top_k=3)

    if not results:
        return json.dumps({"message": "No relevant documents found. Try different search terms."})

    formatted = []
    for score, chunk in results:
        formatted.append({
            "relevance_score": round(score, 2),
            "document": chunk['title'],
            "source": chunk['source'],
            "last_updated": chunk['last_updated'],
            "content": chunk['text']
        })

    return json.dumps(formatted, indent=2)


# Tool definition — tells Claude how to call the search function
my_search_tool = {
    "name": "search_my_docs",
    "description": (
        "Search the knowledge base documents. Returns the most relevant "
        "passages for a query. Always search before answering questions."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query — use specific terms related to the question"
            }
        },
        "required": ["query"]
    }
}

my_tools = [my_search_tool]
my_tool_functions = {"search_my_docs": search_my_docs}

print("✅ search_my_docs() + tool definition ready")
print("   (Will search whatever documents you load in Task 1)")

In [ ]:
# ============================================
# THE RAG AGENT LOOP (from class)
# ============================================
# Same ReAct pattern: THINK → ACT → OBSERVE → REPEAT or CONCLUDE
# Uses MY_DOMAIN variable so the system prompt matches your topic.

def run_rag_agent(user_message: str, tools: list, tool_functions: dict, max_steps: int = 10):
    """
    Run a RAG knowledge agent using the ReAct framework.
    Searches documents before answering. Cites sources.
    """
    system_prompt = (
        f"You are a {MY_DOMAIN} knowledge agent using the ReAct framework.\n"
        f"You answer questions about {MY_DOMAIN} using ONLY the provided documents.\n\n"
        "For every question:\n"
        "1. THINK: What specific information do I need? Which document likely has it?\n"
        "2. ACT: Search documents with specific terms.\n"
        "3. OBSERVE: Did I find the relevant info? Is it current?\n"
        "4. REPEAT or CONCLUDE with a sourced answer.\n\n"
        "CRITICAL RULES:\n"
        "- ALWAYS search documents before answering questions\n"
        "- ONLY state facts found in documents — never make up information\n"
        "- If documents don't contain the answer, say: 'I don't have information on that in our current documents.'\n"
        "- Always cite which document your answer comes from\n"
        "- If you find conflicting information, flag it explicitly and note document dates\n"
        "- Pay attention to document dates — prefer the most recent version"
    )

    print(f"\n{'=' * 60}")
    print(f"💬 User: {user_message}")
    print(f"{'=' * 60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while step < max_steps:
        step += 1

        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        print(f"\n{'- ' * 30}")
        print(f"🔄 Step {step} | Stop reason: {response.stop_reason}")

        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text" and block.text.strip():
                    print(f"\n   💭 THOUGHT:")
                    for line in block.text.strip().split('\n'):
                        print(f"      {line}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    print(f"\n   🎯 ACTION: {tool_name}")
                    print(f"      Input: {json.dumps(tool_input, indent=2)[:200]}")

                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = json.dumps({"error": f"Unknown tool: {tool_name}"})

                    print(f"\n   👁️  OBSERVATION:")
                    result_preview = result[:400] + "..." if len(result) > 400 else result
                    for line in result_preview.split('\n')[:10]:
                        print(f"      {line}")
                    if len(result) > 400:
                        print(f"      [...truncated for display, full data sent to agent]")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            print(f"\n{'=' * 60}")
            print(f"🤖 FINAL ANSWER:")
            print(f"{'=' * 60}")
            print(final_answer)
            print(f"\n✅ Done in {step} step(s)")

            return {"answer": final_answer, "messages": messages, "steps": step}

    print(f"\n⚠️ Max steps ({max_steps}) reached.")
    return {"answer": "Max steps reached.", "messages": messages, "steps": step}

print("✅ run_rag_agent() ready")
print("   Uses MY_DOMAIN variable for the system prompt.")

---

## 📋 Task 1: Build Your Knowledge Agent (45 min)

Pick something you care about. A restaurant, a sports team, a club, a startup idea, your university, a hobby — anything with enough information to fill 5 documents.

| Domain Idea | Example Documents |
|---|---|
| 🍕 Restaurant | Menu, allergen info, hours, catering, reviews |
| ⚽ Sports Team | Roster, schedule, ticket prices, stadium info, history |
| 🚀 Startup | Product features, pricing, FAQ, team bios, roadmap |
| 🎓 University | Admissions, programs, campus life, financial aid, clubs |
| 🏋️ Gym / Studio | Classes, membership tiers, trainers, schedule, policies |
| 🎵 Music Artist | Discography, tour dates, bio, merch, fan FAQ |

### Step 1: Name Your Domain

In [ ]:
# ============================================
# STEP 1: Name your domain
# ============================================
# Replace this with YOUR topic.

MY_DOMAIN = "Mario's Italian Restaurant"  # <-- Change this!

print(f"🎯 Your domain: {MY_DOMAIN}")
print(f"   Your agent will become a {MY_DOMAIN} expert.")

### Step 2: Load Your Documents

Fill in **at least 5 documents** about your domain. Each document needs:
- `title` — what the document is about
- `source` — where it came from (can be made up, like "menu.pdf")
- `last_updated` — when it was last updated
- `content` — the actual text (aim for 100–500 words per document)

**Tips:**
- Quality > quantity. One detailed document beats five vague ones.
- Include specific facts: names, numbers, prices, dates, policies.
- Copy-paste from real sources (websites, menus, brochures) — that’s fine!
- Use AI to help draft content if you want. The learning is in testing, not typing.

**Document 1 is filled in as an example** (a restaurant menu). Replace it with your own content, then fill in the rest.

In [ ]:
# ============================================
# STEP 2: Your documents
# ============================================
# Document 1 is filled in as an example. Replace it with YOUR content,
# then fill in documents 2–5 (or add more!).

my_documents = [
    {
        "title": "Menu — Main Dishes",  # <-- Change this!
        "source": "menu.pdf",
        "last_updated": "2024",
        "content": (
            "Mario's Italian Restaurant — Main Menu\n\n"
            "PASTA:\n"
            "Spaghetti Bolognese ($16) — House-made pasta with slow-cooked beef ragu, "
            "topped with Parmigiano-Reggiano.\n"
            "Fettuccine Alfredo ($15) — Creamy parmesan sauce with fresh fettuccine. "
            "Add grilled chicken +$4, shrimp +$6.\n"
            "Penne Arrabbiata ($14) — Spicy tomato sauce with garlic and red pepper flakes. "
            "Vegan-friendly.\n\n"
            "PIZZA:\n"
            "Classic Margherita ($16) — San Marzano tomatoes, fresh mozzarella, basil.\n"
            "Quattro Formaggi ($19) — Mozzarella, gorgonzola, fontina, parmigiano.\n"
            "Diavola ($18) — Spicy salami, chili oil, mozzarella, fresh basil.\n\n"
            "MAINS:\n"
            "Grilled Branzino ($28) — Mediterranean sea bass with lemon, capers, and "
            "roasted vegetables.\n"
            "Chicken Parmigiana ($22) — Breaded chicken breast, marinara sauce, "
            "melted mozzarella, served with spaghetti.\n\n"
            "All entrees include a house salad or soup of the day. "
            "Gluten-free pasta available +$3."
        )
    },
    {
        "title": "YOUR DOCUMENT 2 TITLE",  # <-- Change this!
        "source": "your-source.pdf",
        "last_updated": "2024",
        "content": (
            # YOUR CONTENT HERE — paste or type your document text.
            # Aim for 100–500 words with specific facts, names, numbers.
            # Example: allergen info, hours, policies, team bios, pricing, etc.
            "Replace this with your document content."
        )
    },
    {
        "title": "YOUR DOCUMENT 3 TITLE",  # <-- Change this!
        "source": "your-source.pdf",
        "last_updated": "2024",
        "content": (
            # YOUR CONTENT HERE — another document about your domain.
            # Could be: FAQ, product info, schedule, policies, history, etc.
            "Replace this with your document content."
        )
    },
    {
        "title": "YOUR DOCUMENT 4 TITLE",  # <-- Change this!
        "source": "your-source.pdf",
        "last_updated": "2024",
        "content": (
            # YOUR CONTENT HERE — different topic from the others.
            # Think: what would a customer/user ask about?
            "Replace this with your document content."
        )
    },
    {
        "title": "YOUR DOCUMENT 5 TITLE",  # <-- Change this!
        "source": "your-source.pdf",
        "last_updated": "2024",
        "content": (
            # YOUR CONTENT HERE — last required document.
            # Tip: include something with numbers (prices, dates, stats)
            # so you can test if the agent retrieves them accurately.
            "Replace this with your document content."
        )
    }
]

print(f"📝 Defined {len(my_documents)} documents for {MY_DOMAIN}")
for i, doc in enumerate(my_documents):
    print(f"   {i+1}. {doc['title']}")

In [ ]:
# ============================================
# PROCESS YOUR DOCUMENTS (just run this)
# ============================================
# Chunks your documents and prepares the search tool.

my_chunks = chunk_documents(my_documents)

print(f"📄 Loaded {len(my_documents)} documents")
print(f"✂️  Created {len(my_chunks)} chunks")
print()
for i, doc in enumerate(my_documents):
    doc_chunks = [c for c in my_chunks if c['title'] == doc['title']]
    print(f"   📝 {doc['title']} → {len(doc_chunks)} chunk(s)")

print(f"\n✅ Ready! Your agent can now search these documents.")

### Step 3: Ask Your Agent Questions!

Try **3 types** of questions:

1. **Clearly answerable** — the answer is directly in one of your documents
2. **Needs multiple documents** — the agent has to search and combine info from different docs
3. **NOT in your documents** — the agent should say "I don’t know"

Replace the example questions below with questions about **your** domain.

In [ ]:
# Question 1: Clearly answerable from your documents
result1 = run_rag_agent(
    "How much does the Spaghetti Bolognese cost?",  # <-- Change this!
    my_tools, my_tool_functions
)

In [ ]:
# Question 2: Needs info from multiple documents
result2 = run_rag_agent(
    "Can I get a gluten-free pizza for a party of 12?",  # <-- Change this!
    my_tools, my_tool_functions
)

In [ ]:
# Question 3: NOT in your documents — agent should say "I don't know"
result3 = run_rag_agent(
    "Do you offer cooking classes on weekends?",  # <-- Change this!
    my_tools, my_tool_functions
)

In [ ]:
# BONUS: Add 2 more questions of your own!

result4 = run_rag_agent(
    "YOUR QUESTION HERE",  # <-- Change this!
    my_tools, my_tool_functions
)

result5 = run_rag_agent(
    "YOUR QUESTION HERE",  # <-- Change this!
    my_tools, my_tool_functions
)

### ✅ Task 1 Checkpoint

Before moving on, check:

- [ ] Did your agent find the right documents for answerable questions?
- [ ] Did it cite which document the answer came from?
- [ ] Did it say "I don’t know" for the question NOT in your documents?

If yes — **Task 1 done!** 🎉

---

## 💥 Task 2: Break Your Agent (30 min)

Knowing **where** AI breaks is more valuable than knowing where it works. In this task, you’ll deliberately create situations that trip up your agent.

**Three failure modes to test:**

1. **Outdated Data** — Add an old version of a document with different facts. Which version does the agent use?
2. **Contradicting Documents** — Add a document that says the opposite of an existing one. How does the agent handle the conflict?
3. **Knowledge Gap** — Ask about something that’s plausible but NOT in any document. Does the agent admit it doesn’t know, or hallucinate?

In [ ]:
# ============================================
# FAILURE MODE 1: Outdated Data
# ============================================
# Add an OLD version of one of your documents with different facts.
# Example: an old menu with different prices, or an old policy.

outdated_doc = {
    "title": "Menu — Main Dishes",  # <-- Same title as your current doc!
    "source": "menu-2022-archive.pdf",
    "last_updated": "2022",  # <-- Older date
    "content": (
        # OLD content with DIFFERENT facts (e.g., different prices)
        "Mario's Italian Restaurant — Main Menu (2022)\n\n"
        "PASTA:\n"
        "Spaghetti Bolognese ($12) — House-made pasta with beef ragu.\n"  # Was $16!
        "Fettuccine Alfredo ($11) — Creamy parmesan sauce.\n"  # Was $15!
        "\nPIZZA:\n"
        "Classic Margherita ($12) — Tomatoes, mozzarella, basil.\n"  # Was $16!
    )
}

# Rebuild with the outdated doc included
docs_with_outdated = my_documents + [outdated_doc]
my_chunks = chunk_documents(docs_with_outdated)
print(f"⚠️ Rebuilt index with {len(docs_with_outdated)} docs ({len(my_chunks)} chunks)")
print(f"   Includes outdated document from {outdated_doc['last_updated']}")

# Ask a question that hits the conflict
print("\n🧪 Testing: Does the agent use the old price or the new price?")
result_outdated = run_rag_agent(
    "How much does the Spaghetti Bolognese cost?",  # <-- Change to match YOUR docs
    my_tools, my_tool_functions
)

In [ ]:
# ============================================
# FAILURE MODE 2: Contradicting Documents
# ============================================
# Add a document that directly contradicts an existing one.
# Example: two different allergy policies, or two different prices.

contradicting_doc = {
    "title": "Allergen Policy Update",  # <-- Change this!
    "source": "allergen-update-memo.pdf",
    "last_updated": "2024",  # Same year — no date difference to rely on
    "content": (
        # Content that CONTRADICTS an existing document
        # Replace with something relevant to YOUR domain
        "IMPORTANT UPDATE: Due to supplier changes, Mario's can NO LONGER "
        "accommodate gluten-free requests. All pasta and pizza dough contains "
        "wheat gluten. We apologize for the inconvenience."
    )
}

# Rebuild with the contradiction included
docs_with_contradiction = my_documents + [contradicting_doc]
my_chunks = chunk_documents(docs_with_contradiction)
print(f"⚠️ Rebuilt index with {len(docs_with_contradiction)} docs ({len(my_chunks)} chunks)")
print(f"   Includes contradicting document: {contradicting_doc['title']}")

# Ask a question that triggers the contradiction
print("\n🧪 Testing: How does the agent handle contradicting info?")
result_contradiction = run_rag_agent(
    "Can I get a gluten-free meal?",  # <-- Change to match YOUR contradiction
    my_tools, my_tool_functions
)

In [ ]:
# ============================================
# FAILURE MODE 3: Knowledge Gap
# ============================================
# Ask about something plausible but NOT in any document.
# A good agent says "I don't know." A bad one makes something up.

# Reset to original documents (remove outdated + contradicting)
my_chunks = chunk_documents(my_documents)
print(f"✅ Reset to original {len(my_documents)} documents\n")

# Ask questions about things NOT in your documents
print("🧪 Testing: Does the agent admit it doesn't know?")

gap_questions = [
    "What's the WiFi password?",  # <-- Change these to YOUR domain!
    "Who is the head chef and where did they train?",
    "Do you have a loyalty rewards program?",
]

for q in gap_questions:
    result_gap = run_rag_agent(q, my_tools, my_tool_functions)
    print("\n" + "-" * 40)

### 📝 Write Up Your Findings

**Double-click this cell to edit it.** Fill in your observations for each failure mode.

---

**Failure Mode 1: Outdated Data**

- What I tested: *(describe your outdated document and question)*
- What happened: *(did the agent use the old or new info?)*
- Why it’s a problem: *(what could go wrong in a real business?)*
- How I’d fix it: *(e.g., add timestamps, remove old docs, date-aware search)*

**Failure Mode 2: Contradicting Documents**

- What I tested: *(describe the contradiction)*
- What happened: *(did the agent flag the conflict or pick one silently?)*
- Why it’s a problem: *(what if a customer relied on this answer?)*
- How I’d fix it: *(e.g., version control, conflict detection, human review)*

**Failure Mode 3: Knowledge Gap**

- What I tested: *(describe your unanswerable questions)*
- What happened: *(did the agent say "I don't know" or hallucinate?)*
- Why it’s a problem: *(what if the agent confidently gives a wrong answer?)*
- How I’d fix it: *(e.g., confidence thresholds, fallback to human, guardrails)*

### ✅ Task 2 Checkpoint

- [ ] Tested all 3 failure modes (outdated, contradiction, gap)
- [ ] Wrote up findings for each one
- [ ] Proposed at least one fix per failure mode

**Task 2 done!** 🎉

---

## 📊 Task 3: Score Your Agent (30 min)

Time to be a scientist. You’ll run **10 test questions** through your agent and score each answer on 4 criteria.

**Scoring Criteria:**

| Criteria | What It Means |
|----------|---------------|
| **Right Doc?** | Did the agent search and find the correct document? |
| **Correct?** | Is the answer factually accurate based on your documents? |
| **Hallucinated?** | Did the agent make up any information NOT in the docs? |
| **Cited?** | Did the agent say which document the answer came from? |

**Question mix:** Aim for ~5 answerable, ~3 partially answerable, ~2 unanswerable.

In [ ]:
# ============================================
# YOUR 10 TEST QUESTIONS
# ============================================
# Make sure you're back to your original documents.
my_chunks = chunk_documents(my_documents)
print(f"✅ Using original {len(my_documents)} documents ({len(my_chunks)} chunks)\n")

test_questions = [
    # -- Answerable (5 questions) --
    "How much does the Margherita pizza cost?",         # <-- Change these!
    "What are your hours on Saturday?",
    "Do you offer vegan options?",
    "How do I make a reservation for a large group?",
    "What comes with the entrees?",
    # -- Partially answerable (3 questions) --
    "Compare your cheapest and most expensive dishes.",
    "What's the best option for someone with nut allergies?",
    "Can I host a 50-person wedding reception?",
    # -- Unanswerable (2 questions) --
    "What wines do you recommend with the salmon?",
    "Does the restaurant have outdoor seating?",
]

# Run all 10 questions
results = []
for i, question in enumerate(test_questions, 1):
    print(f"\n{'#' * 60}")
    print(f"# QUESTION {i} of {len(test_questions)}")
    print(f"{'#' * 60}")
    result = run_rag_agent(question, my_tools, my_tool_functions)
    results.append(result)

print(f"\n\n✅ All {len(test_questions)} questions completed!")

### 📝 Scorecard

**Double-click this cell to edit.** Fill in the table based on the output above.

| # | Question | Right Doc? | Correct? | Hallucinated? | Cited? |
|---|----------|:----------:|:--------:|:-------------:|:------:|
| 1 | *(your question)* | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 2 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 3 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 4 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 5 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 6 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 7 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 8 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 9 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |
| 10 | | ✅/❌ | ✅/❌ | ✅/❌ | ✅/❌ |

> **Reading the table:** ✅ = yes, ❌ = no. For "Hallucinated?" — ❌ is what you *want* (no hallucination). For the others, ✅ is what you want.

**My agent scored ___/10 on correctness.**

### Reflection

**Double-click to edit.** Answer these questions based on your scorecard:

- What patterns do you see? When does your agent do well vs. struggle?
- *(your answer)*

- Were there any surprising results (good or bad)?
- *(your answer)*

- If you could change one thing about your documents to improve accuracy, what would it be?
- *(your answer)*

---

### ✅ Task 3 Checkpoint

- [ ] Ran all 10 test questions with visible output
- [ ] Filled in the scorecard table
- [ ] Wrote reflections on patterns and surprises

**Task 3 done!** 🎉

---

## 🏢 Task 4: Design a Real-World Knowledge Agent (20 min)

A company hires you as an AI consultant. They want a knowledge agent for their team. What would you build?

Pick a **real company or organization** (or make one up) and design a knowledge agent for them. Think about: What documents would it need? Who would use it? What happens if it’s wrong?

### 📝 Your Proposal

**Double-click this cell to edit.** Fill in each section.

---

**Company/Organization:** *(name and what they do)*

**Use Case:** *(who uses the agent and for what?)*

**Documents Needed (list 5–10):**
1. *(document name — what it contains)*
2. 
3. 
4. 
5. 

**Target Users:** *(employees? customers? both?)*

**Sample Questions the Agent Should Answer (list 3):**
1. 
2. 
3. 

**Risk Assessment:**
- Risk level: LOW / MEDIUM / HIGH
- Worst case if the agent is wrong: *(what happens?)*
- Mitigation: *(how would you reduce this risk?)*

**Build vs. Buy:**
- Would you build a custom RAG system or use an existing platform (like Glean, Notion AI, etc.)?
- Why?

**Success Metrics:**
- How would you measure if this agent is working well?
- *(e.g., accuracy rate, user satisfaction, tickets deflected, time saved)*

### 📚 Example Proposal: Marriott Hotels Front Desk Agent

Here’s what a strong proposal looks like. Use this as a reference.

---

**Company/Organization:** Marriott International — global hotel chain with 8,000+ properties.

**Use Case:** A knowledge agent that helps front desk staff answer guest questions instantly, without flipping through binders or calling a manager.

**Documents Needed:**
1. Property-specific info (room types, amenities, floor plans, checkout times)
2. Loyalty program rules (Marriott Bonvoy tiers, points earning/redemption)
3. Local area guide (restaurants, attractions, transportation, emergency services)
4. Policies (cancellation, pet, smoking, noise complaints, late checkout)
5. Current promotions and packages (seasonal deals, corporate rates)
6. Restaurant/bar menus and hours (on-site dining)
7. Event/conference room details (capacity, AV equipment, catering options)
8. Emergency procedures (fire, medical, severe weather, security incidents)

**Target Users:** Front desk staff, concierge, phone operators (not guests directly).

**Sample Questions:**
1. "A Bonvoy Platinum member wants to know if they qualify for a suite upgrade — we’re at 80% occupancy tonight."
2. "Guest is asking about gluten-free options at our restaurant. What can we offer?"
3. "Someone wants to cancel their reservation for next week. What’s the cancellation window?"

**Risk Assessment:**
- Risk level: **MEDIUM**
- Worst case: Agent quotes an expired promotion or wrong cancellation policy → guest is overcharged or undercharged, brand damage.
- Mitigation: Daily document sync, flag answers older than 30 days, always show "verify with manager" for financial decisions.

**Build vs. Buy:**
- **Buy** — Use an enterprise platform (e.g., Glean or a hospitality-specific solution). Marriott has 8,000 properties, each with different docs. Custom RAG would be a maintenance nightmare. Need a platform that supports per-property document sets with centralized corporate policies.

**Success Metrics:**
- Answer accuracy: >90% on weekly spot-checks
- Time to answer: <30 seconds (vs. 3–5 minutes looking it up manually)
- Staff satisfaction: quarterly survey score >4/5
- Manager escalations reduced by 25% within 3 months

---

## 🎯 Submission Checklist

Before submitting, make sure:

- [ ] **Task 1:** Domain chosen, 5+ documents loaded, 3+ test questions run with visible output
- [ ] **Task 2:** All 3 failure modes tested, findings written up
- [ ] **Task 3:** 10 test questions run, scorecard filled in, reflection written
- [ ] **Task 4:** Real-world proposal completed (all sections filled)
- [ ] All code cells have been run (output is visible)
- [ ] Sharing: File → Share → "Anyone with the link can view"

**Paste your Colab link in Canvas. You’re done!** 🎉

---

## You Just Built a Knowledge Agent for YOUR Domain

Not a tutorial. Not a demo. **Your** agent, **your** documents, **your** expertise.

You loaded real information, tested whether the agent could use it accurately, found where it breaks, and designed how you'd deploy something like this in the real world. That's the full lifecycle of an AI project.

### The Stack So Far

```
Session 4: Tools          → Claude can DO things (calculator, weather)
Session 5: Real APIs      → Claude connects to the real world (Wikipedia, web)
Session 6: ReAct          → Claude can REASON through multi-step problems
Session 7: RAG            → Claude reads YOUR documents (today's homework!)
```

**Next up — Session 8: Multi-Agent Systems.** One agent is useful. A *team* of agents working together is powerful.

---

*ISOM 260: AI for Business* | [Course Website](https://isom-260.vercel.app)